# Facepoints inference

This notebook is independent of the training workflow. It loads a saved checkpoint and `outputs/normalization.json`, predicts 14 facial landmarks for every supported image in the selected directory, displays examples and writes a CSV with coordinates in the original image space.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
from PIL import Image

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

IMAGES_DIR = PROJECT_ROOT / "data" / "public_tests" / "00_test_img_input" / "test" / "images"
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "facepoints_model.pt"
NORMALIZATION_PATH = PROJECT_ROOT / "outputs" / "normalization.json"
PREDICTIONS_PATH = PROJECT_ROOT / "outputs" / "inference_predictions.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256
N_IMAGES_TO_DISPLAY = 4

from facepoints.evaluate import FacepointsEvaluator
from facepoints.utils import save_predictions
from facepoints.visualization import visualize_keypoints

if not IMAGES_DIR.is_dir():
    raise FileNotFoundError(f"Set IMAGES_DIR to an existing image directory: {IMAGES_DIR}")
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
if not NORMALIZATION_PATH.is_file():
    raise FileNotFoundError("Run notebooks/exploration.ipynb to create outputs/normalization.json.")

print(f"Device: {DEVICE}")
print(f"Images: {IMAGES_DIR}")
print(f"Checkpoint: {CHECKPOINT_PATH}")


## Load the model and run inference


In [ ]:
evaluator = FacepointsEvaluator(
    checkpoint_path=CHECKPOINT_PATH,
    normalization_path=NORMALIZATION_PATH,
    device=DEVICE,
)
predictions = evaluator.predict(IMAGES_DIR, batch_size=BATCH_SIZE)
print(f"Predicted images: {len(predictions):,}")
print(f"Keypoints per image: {len(next(iter(predictions.values()))) // 2 if predictions else 14}")


## Inspect predicted coordinates


In [ ]:
for image_name, coordinates in list(predictions.items())[:5]:
    print(image_name, coordinates.reshape(14, 2))


## Visualise predictions


In [ ]:
for image_name in list(predictions)[:N_IMAGES_TO_DISPLAY]:
    with Image.open(IMAGES_DIR / image_name) as source:
        image = source.convert("RGB")
    figure, _ = visualize_keypoints(
        image,
        prediction=predictions[image_name],
        title=f"Prediction: {image_name}",
        show_indices=True,
    )
    plt.show()
    plt.close(figure)


## Save predictions

The CSV contains the image filename followed by 28 coordinate values. Coordinates are expressed in the original image pixel space, not in the resized model-input space.


In [ ]:
save_predictions(PREDICTIONS_PATH, predictions)
print(f"Predictions saved to: {PREDICTIONS_PATH}")
